<a href="https://colab.research.google.com/github/ZarmelZar/ZarmelZar/blob/main/Project_Comp_215_Melika_Mousavikhah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [57]:
!pip install openrouteservice folium
!pip install polyline
import requests
import folium
from openrouteservice import convert
from IPython.display import display
import json
import polyline

In [58]:
class DataLoader:

    def __init__(self):
        self.geolocator_url = "https://nominatim.openstreetmap.org/search"
        self.api_url = "https://api.openrouteservice.org/v2/directions/driving-car"
        self.api_key = "5b3ce3597851110001cf62486fa8e28a936a46708e15a25f10f55adc"

    def get_coordinates(self, location_name):
        params_dict = {'q': location_name, 'format': 'json'}
        headers = {"User-Agent": "MyPathfinderApp/1.0 (Melikamsk122@gmail.com)"}

        response = requests.get(self.geolocator_url, params=params_dict, headers=headers)

        if response.status_code == 200:
            try:
                data = response.json()

                if len(data) > 0:
                    coords = [data[0]['lon'], data[0]['lat']]  # گرفتن مختصات longitude و latitude
                    print(f"Latitude: {coords[1]}, Longitude: {coords[0]}")
                    return {'latitude': coords[1], 'longitude': coords[0]}
                else:
                    print(f"No results found for {location_name}.")
                    return None
            except ValueError as e:
                print(f"Error decoding JSON: {e}")
                print(f"Response text: {response.text}")
                return None
        else:
            print(f"Error loading data from OpenStreetMap: {response.status_code}")
            return None

    def get_route(self, start_coords, end_coords):
        headers = {'Authorization': self.api_key, 'Content-Type': 'application/json'}

        body = {
            "coordinates": [
                [start_coords['longitude'], start_coords['latitude']],
                [end_coords['longitude'], end_coords['latitude']]
            ]
        }

        response = requests.post(self.api_url, json=body, headers=headers)


        if response.status_code == 200:
            try:
                data = response.json()

                if 'routes' in data and len(data['routes']) > 0:
                    route = []
                    geometry = data['routes'][0]['geometry']
                    decoded_route = polyline.decode(geometry)  # دیکد کردن پلی‌لاین

                    # افزودن مختصات مرحله به مرحله
                    for segment in data['routes'][0]['segments']:
                        for step in segment['steps']:
                            route.append((step['instruction'], step['distance'], step['duration']))

                    print(route)
                    return route, decoded_route
                else:
                    print("No route found.")
                    return None, None
            except ValueError as e:
                print(f"Error decoding JSON: {e}")
                print(f"Response text: {response.text}")
                return None, None
        else:
            print(f"Error loading route data: {response.status_code}")
            return None, None

    def get_user_location(self):
        # گرفتن آدرس از کاربر برای لوکیشن فعلی
        start_address = input("Enter your current location address: ")
        start_coords = self.get_coordinates(start_address)

        # گرفتن آدرس از کاربر برای مقصد
        destination_address = input("Enter the destination address: ")
        destination_coords = self.get_coordinates(destination_address)

        if start_coords and destination_coords:
            return start_coords, destination_coords
        else:
            print("Could not find coordinates for the given addresses.")
            return None, None

In [59]:
class MapVisualizer:
    def __init__(self, start_coords, destination_coords, route, decoded_route):
        self.start_coords = start_coords
        self.destination_coords = destination_coords
        self.route = route
        self.decoded_route = decoded_route

    def create_map(self):
        # ایجاد نقشه
        m = folium.Map(location=[self.start_coords['latitude'], self.start_coords['longitude']], zoom_start=14)

        # افزودن نشانه برای مبدا
        folium.Marker([self.start_coords['latitude'], self.start_coords['longitude']], popup="Start").add_to(m)

        # افزودن نشانه برای مقصد
        folium.Marker([self.destination_coords['latitude'], self.destination_coords['longitude']], popup="Destination").add_to(m)

        # نمایش خط مسیر
        folium.PolyLine(self.decoded_route, color="blue", weight=2.5, opacity=1).add_to(m)

        # نمایش نقشه
        display(m)

In [67]:
dl = DataLoader()

# گرفتن موقعیت و مقصد از کاربر
start_coords, destination_coords = dl.get_user_location()

if start_coords and destination_coords:
    # دریافت مسیر و دیکد کردن پلی‌لاین
    route, decoded_route = dl.get_route(start_coords, destination_coords)

    if route and decoded_route:
        # نمایش نقشه
        map_visualizer = MapVisualizer(start_coords, destination_coords, route, decoded_route)
        map_visualizer.create_map()
    else:
        print("No route found to display on map.")
else:
    print("Could not find coordinates for the given addresses.")

Enter your current location address: 2008 fullerton ave
Latitude: 49.330807199999995, Longitude: -123.12104216138323
Enter the destination address: 2055 purcell way north vancouver
Latitude: 49.317583189725674, Longitude: -123.02098513864625
[('Head southwest on Keith Road', 202.1, 48.5), ('Turn sharp right onto 3rd Street', 1163.9, 113.8), ('Turn right onto Taylor Way, 99, 1A', 246.0, 25.6), ('Keep right', 7656.5, 526.6), ('Keep right', 826.7, 84.3), ('Turn left onto East Keith Road', 1189.5, 175.5), ('Turn left onto Lillooet Road', 599.0, 84.5), ('Turn right onto Purcell Way', 233.3, 56.0), ('Turn sharp left onto Purcell Way', 24.9, 6.0), ('Arrive at Purcell Way, straight ahead', 0.0, 0.0)]
